# Coastal ecosystem connectivity: tri-ecosystem proximity, protected areas, and avoided EAD linkage

This notebook extends the mangrove patch proximity analysis by asking two additional questions:

- where do **mangrove–coral–seagrass connected seascapes** occur at **250 m**, **500 m**, and **1000 m**?
- to what extent are these connected seascapes, and the mangrove patches within them, already covered by **protected areas**?

The notebook works from the patch-level connectivity and avoided-EAD outputs generated in `coastal_ecosystem_connectivity_mangrove_patch_proximity_ead_linkage.ipynb` and adds protected-area linkage.

It produces:
- a mangrove patch table with protected-area status added
- threshold-specific tri-ecosystem proximity zone polygons
- area summaries showing how much of the tri-ecosystem zone is inside protected areas
- patch summaries showing how many **positive-net** mangrove patches near both coral and seagrass are protected, and how much avoided EAD they account for
- candidate tables of **unprotected positive-net** mangrove patches within tri-ecosystem settings


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from IPython.display import display

pd.options.display.float_format = "{:,.2f}".format

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
patch_proximity_gpkg = (
    base_path
    / "dphil_paper_3/results/co_benefits/connectivity/coastal_connectivity/mangrove_patch_proximity_to_coral_seagrass_and_ead_linkage.gpkg"
)
coral_path = base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/corals/Coral Reefs.shp"
seagrass_path = base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/seagrass/Seagrass.shp"
protected_areas_path = (
    base_path / "dphil_common_cross_cutting/common_incoming_data/protected_landcover/protected_areas.shp"
)
output_dir = base_path / "dphil_paper_3/results/co_benefits/connectivity/coastal_connectivity"
output_dir.mkdir(parents=True, exist_ok=True)

jamaica_metric_grid_crs = "EPSG:3448"
proximity_thresholds_m = [250, 500, 1000]
scenario_names = ["minimum", "maximum"]


## Inputs and scope

- The mangrove patch table is taken from the patch-level connectivity notebook and therefore already contains:
  - mangrove patch geometry
  - nearest coral and seagrass distances
  - 250 m, 500 m, and 1000 m proximity flags
  - minimum- and maximum-scenario patch-level net avoided EADs
- Coral reefs and seagrass are taken from the current common incoming land-cover layers.
- Protected-area linkage is based on the current `protected_areas.shp` layer.

Here, **tri-ecosystem proximity** refers to mangrove patches that are within the chosen distance threshold of **both** coral reefs and seagrass, and to the corresponding threshold-specific spatial zones created from the intersection of buffered mangrove, coral, and seagrass layers.


In [ ]:
def load_geodataframe(input_path, target_crs):
    geodataframe = gpd.read_file(input_path)
    if geodataframe.crs is None:
        raise ValueError(f"CRS is missing for {input_path}")
    if str(geodataframe.crs).upper() != target_crs:
        geodataframe = geodataframe.to_crs(target_crs)
    return geodataframe


def attach_protected_area_status(mangrove_patch_table, protected_areas):
    joined_table = gpd.sjoin(
        mangrove_patch_table[["mangrove_patch_id", "geometry"]],
        protected_areas[["NAME", "geometry"]],
        how="left",
        predicate="intersects",
    )

    protected_area_summary = (
        joined_table.groupby("mangrove_patch_id", as_index=False)
        .agg(
            protected_area_count=(
                "NAME",
                lambda values: int(pd.Series(values).dropna().astype(str).nunique()),
            ),
            protected_area_name_list=(
                "NAME",
                lambda values: "; ".join(sorted(pd.Series(values).dropna().astype(str).unique())),
            ),
        )
    )

    output_table = mangrove_patch_table.merge(
        protected_area_summary,
        on="mangrove_patch_id",
        how="left",
    )
    output_table["protected_area_count"] = output_table["protected_area_count"].fillna(0).astype(int)
    output_table["protected_area_name_list"] = output_table["protected_area_name_list"].fillna("")
    output_table["intersects_protected_area"] = output_table["protected_area_count"] > 0
    return output_table


def build_tri_ecosystem_zone(mangrove_patches, corals, seagrass, distance_m):
    mangrove_buffer_union = mangrove_patches.geometry.buffer(distance_m).union_all()
    coral_buffer_union = corals.geometry.buffer(distance_m).union_all()
    seagrass_buffer_union = seagrass.geometry.buffer(distance_m).union_all()
    tri_ecosystem_zone_geometry = mangrove_buffer_union.intersection(coral_buffer_union).intersection(seagrass_buffer_union)
    return tri_ecosystem_zone_geometry


def build_tri_zone_record(distance_m, tri_ecosystem_zone_geometry, protected_areas, protected_area_union_geometry):
    tri_zone_area_ha = tri_ecosystem_zone_geometry.area / 10_000 if not tri_ecosystem_zone_geometry.is_empty else 0.0
    tri_zone_protected_area_ha = (
        tri_ecosystem_zone_geometry.intersection(protected_area_union_geometry).area / 10_000
        if not tri_ecosystem_zone_geometry.is_empty
        else 0.0
    )
    tri_zone_unprotected_area_ha = max(tri_zone_area_ha - tri_zone_protected_area_ha, 0.0)

    if tri_ecosystem_zone_geometry.is_empty:
        intersecting_protected_area_names = []
    else:
        intersecting_protected_area_names = sorted(
            protected_areas.loc[
                protected_areas.geometry.intersects(tri_ecosystem_zone_geometry),
                "NAME",
            ].dropna().astype(str).unique()
        )

    return {
        "distance_m": distance_m,
        "tri_zone_area_ha": tri_zone_area_ha,
        "tri_zone_protected_area_ha": tri_zone_protected_area_ha,
        "tri_zone_unprotected_area_ha": tri_zone_unprotected_area_ha,
        "tri_zone_share_protected_pct": (
            100.0 * tri_zone_protected_area_ha / tri_zone_area_ha if tri_zone_area_ha > 0 else 0.0
        ),
        "intersecting_protected_area_count": len(intersecting_protected_area_names),
        "intersecting_protected_area_name_list": "; ".join(intersecting_protected_area_names),
        "geometry": tri_ecosystem_zone_geometry,
    }


def build_patch_group_near_both_protection_summary(mangrove_patch_table, scenario_name, thresholds_m):
    net_avoided_ead_column = f"net_avoided_ead_usd_{scenario_name}"
    group_filters = {
        "positive_net": mangrove_patch_table[net_avoided_ead_column] > 0,
        "non_positive_net": mangrove_patch_table[net_avoided_ead_column] <= 0,
    }

    summary_rows = []
    for distance_m in thresholds_m:
        both_column = f"both_within_{distance_m}m"
        for patch_group, group_filter in group_filters.items():
            total_group_patch_count = int(group_filter.sum())
            group_near_both = mangrove_patch_table.loc[group_filter & mangrove_patch_table[both_column]].copy()
            group_near_both_patch_count = len(group_near_both)
            group_near_both_protected = group_near_both.loc[group_near_both["intersects_protected_area"]].copy()
            group_near_both_unprotected = group_near_both.loc[~group_near_both["intersects_protected_area"]].copy()
            total_group_near_both_net_ead_usd = float(group_near_both[net_avoided_ead_column].sum())
            protected_group_near_both_net_ead_usd = float(group_near_both_protected[net_avoided_ead_column].sum())
            unprotected_group_near_both_net_ead_usd = float(group_near_both_unprotected[net_avoided_ead_column].sum())

            summary_rows.append(
                {
                    "scenario": scenario_name,
                    "distance_m": distance_m,
                    "patch_group": patch_group,
                    "total_group_patch_count": total_group_patch_count,
                    "group_near_both_patch_count": group_near_both_patch_count,
                    "share_of_group_near_both_pct": (
                        100.0 * group_near_both_patch_count / total_group_patch_count if total_group_patch_count > 0 else 0.0
                    ),
                    "group_near_both_protected_patch_count": len(group_near_both_protected),
                    "group_near_both_unprotected_patch_count": len(group_near_both_unprotected),
                    "share_of_group_near_both_protected_patch_count_pct": (
                        100.0 * len(group_near_both_protected) / group_near_both_patch_count if group_near_both_patch_count > 0 else 0.0
                    ),
                    "share_of_group_near_both_unprotected_patch_count_pct": (
                        100.0 * len(group_near_both_unprotected) / group_near_both_patch_count if group_near_both_patch_count > 0 else 0.0
                    ),
                    "group_near_both_net_ead_usd": total_group_near_both_net_ead_usd,
                    "group_near_both_protected_net_ead_usd": protected_group_near_both_net_ead_usd,
                    "group_near_both_unprotected_net_ead_usd": unprotected_group_near_both_net_ead_usd,
                    "share_of_group_near_both_protected_net_ead_pct": (
                        100.0 * protected_group_near_both_net_ead_usd / total_group_near_both_net_ead_usd
                        if total_group_near_both_net_ead_usd != 0
                        else 0.0
                    ),
                    "share_of_group_near_both_unprotected_net_ead_pct": (
                        100.0 * unprotected_group_near_both_net_ead_usd / total_group_near_both_net_ead_usd
                        if total_group_near_both_net_ead_usd != 0
                        else 0.0
                    ),
                }
            )
    return pd.DataFrame(summary_rows)


def build_unprotected_positive_net_candidate_table(mangrove_patch_table, scenario_names, thresholds_m):
    candidate_parts = []
    candidate_columns = [
        "mangrove_patch_id",
        "parish",
        "area_ha",
        "protected_area_count",
        "protected_area_name_list",
        "nearest_coral_distance_m",
        "nearest_seagrass_distance_m",
    ]
    for scenario_name in scenario_names:
        net_avoided_ead_column = f"net_avoided_ead_usd_{scenario_name}"
        for distance_m in thresholds_m:
            both_column = f"both_within_{distance_m}m"
            candidate_table = mangrove_patch_table.loc[
                (mangrove_patch_table[net_avoided_ead_column] > 0)
                & mangrove_patch_table[both_column]
                & (~mangrove_patch_table["intersects_protected_area"]),
                candidate_columns,
            ].copy()
            candidate_table["scenario"] = scenario_name
            candidate_table["distance_m"] = distance_m
            candidate_table["net_avoided_ead_usd"] = mangrove_patch_table.loc[candidate_table.index, net_avoided_ead_column].values
            candidate_parts.append(candidate_table)

    if not candidate_parts:
        return pd.DataFrame()

    return pd.concat(candidate_parts, ignore_index=True).sort_values(
        ["scenario", "distance_m", "net_avoided_ead_usd", "mangrove_patch_id"],
        ascending=[True, True, False, True],
    ).reset_index(drop=True)


In [ ]:
mangrove_patch_proximity = load_geodataframe(patch_proximity_gpkg, jamaica_metric_grid_crs)
corals = load_geodataframe(coral_path, jamaica_metric_grid_crs)
seagrass = load_geodataframe(seagrass_path, jamaica_metric_grid_crs)
protected_areas = load_geodataframe(protected_areas_path, jamaica_metric_grid_crs)
protected_area_union_geometry = protected_areas.geometry.union_all()

mangrove_patch_proximity = attach_protected_area_status(mangrove_patch_proximity, protected_areas)

print(f"Mangrove patches loaded: {len(mangrove_patch_proximity)}")
print(f"Protected areas loaded: {len(protected_areas)}")
display(mangrove_patch_proximity.drop(columns=["geometry"]).head())


In [ ]:
tri_zone_records = []
for distance_m in proximity_thresholds_m:
    tri_ecosystem_zone_geometry = build_tri_ecosystem_zone(
        mangrove_patch_proximity,
        corals,
        seagrass,
        distance_m,
    )
    tri_zone_records.append(
        build_tri_zone_record(
            distance_m,
            tri_ecosystem_zone_geometry,
            protected_areas,
            protected_area_union_geometry,
        )
    )

tri_ecosystem_zone_gdf = gpd.GeoDataFrame(tri_zone_records, geometry="geometry", crs=jamaica_metric_grid_crs)
tri_ecosystem_zone_summary = tri_ecosystem_zone_gdf.drop(columns=["geometry"]).copy()

patch_group_near_both_protection_summary = pd.concat(
    [
        build_patch_group_near_both_protection_summary(
            mangrove_patch_proximity,
            scenario_name,
            proximity_thresholds_m,
        )
        for scenario_name in scenario_names
    ],
    ignore_index=True,
)

unprotected_positive_net_candidates = build_unprotected_positive_net_candidate_table(
    mangrove_patch_proximity,
    scenario_names,
    proximity_thresholds_m,
)

patch_output_csv = output_dir / "mangrove_patch_proximity_protected_area_ead_linkage.csv"
patch_output_gpkg = output_dir / "mangrove_patch_proximity_protected_area_ead_linkage.gpkg"
tri_zone_summary_csv = output_dir / "tri_ecosystem_proximity_zone_protected_area_summary.csv"
tri_zone_gpkg = output_dir / "tri_ecosystem_proximity_zones.gpkg"
patch_group_summary_csv = output_dir / "mangrove_patch_group_near_both_protected_area_summary.csv"
candidate_csv = output_dir / "unprotected_positive_net_mangrove_patches_near_both.csv"

mangrove_patch_proximity.drop(columns=["geometry"]).to_csv(patch_output_csv, index=False)
mangrove_patch_proximity.to_file(patch_output_gpkg, driver="GPKG")
tri_ecosystem_zone_summary.to_csv(tri_zone_summary_csv, index=False)
tri_ecosystem_zone_gdf.to_file(tri_zone_gpkg, driver="GPKG")
patch_group_near_both_protection_summary.to_csv(patch_group_summary_csv, index=False)
unprotected_positive_net_candidates.to_csv(candidate_csv, index=False)

print("Saved outputs:")
for output_path in [
    patch_output_csv,
    patch_output_gpkg,
    tri_zone_summary_csv,
    tri_zone_gpkg,
    patch_group_summary_csv,
    candidate_csv,
]:
    print(f" - {output_path}")

print("Tri-ecosystem zone protected-area summary:")
display(tri_ecosystem_zone_summary)

print("Patch-group near-both protected-area summary:")
display(patch_group_near_both_protection_summary)


In [ ]:
positive_near_both_summary = patch_group_near_both_protection_summary.loc[
    patch_group_near_both_protection_summary["patch_group"] == "positive_net"
].copy()

print("Positive-net mangrove patches near both coral reefs and seagrass:")
display(positive_near_both_summary)

print("Top unprotected positive-net mangrove patches near both ecosystems at 500 m:")
display(
    unprotected_positive_net_candidates.loc[
        unprotected_positive_net_candidates["distance_m"] == 500
    ].head(20)
)
